# 🏙️ Delhi Deep Dive — Module 06: Composite Score
## Weighted multi-hazard composite risk score for all 11 Delhi districts

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

OUTPUT_DIR = '../data/outputs/delhi'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

DELHI_WEIGHTS = {
    'drought_risk_score':       0.15,
    'flood_risk_score':         0.25,
    'heatwave_risk_score':      0.20,
    'airquality_risk_score':    0.25,
    'waterscarcity_risk_score': 0.15,
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

# Load all 5 module outputs
drought  = pd.read_csv(f'{OUTPUT_DIR}/delhi_drought_scores.csv')[['district','year','drought_risk_score']]
flood    = pd.read_csv(f'{OUTPUT_DIR}/delhi_flood_scores.csv')[['district','year','flood_risk_score']]
heatwave = pd.read_csv(f'{OUTPUT_DIR}/delhi_heatwave_scores.csv')[['district','year','heatwave_risk_score']]
airqual  = pd.read_csv(f'{OUTPUT_DIR}/delhi_airquality_scores.csv')[['district','year','airquality_risk_score']]
watersca = pd.read_csv(f'{OUTPUT_DIR}/delhi_waterscarcity_scores.csv')[['district','year','waterscarcity_risk_score']]

comp = drought.merge(flood, on=['district','year'], how='outer')
comp = comp.merge(heatwave, on=['district','year'], how='outer')
comp = comp.merge(airqual,  on=['district','year'], how='outer')
comp = comp.merge(watersca, on=['district','year'], how='outer')

score_cols = list(DELHI_WEIGHTS.keys())
for c in score_cols:
    comp[c] = pd.to_numeric(comp[c], errors='coerce').fillna(0)

comp['delhi_composite_score']    = sum(comp[col]*w for col,w in DELHI_WEIGHTS.items()).round(2)
comp['delhi_composite_category'] = comp['delhi_composite_score'].apply(score_to_category)
for d, c in DELHI_DISTRICTS.items():
    comp.loc[comp['district']==d, 'lat'] = c['lat']
    comp.loc[comp['district']==d, 'lon'] = c['lon']

comp.to_csv(f'{OUTPUT_DIR}/delhi_composite_scores.csv', index=False)
print(f'Saved delhi_composite_scores.csv ({len(comp)} rows)')
print(comp[comp['year']==2023][['district','delhi_composite_score','delhi_composite_category']].sort_values('delhi_composite_score',ascending=False).to_string())

## Composite Heatmap

In [ ]:
latest = comp.sort_values('year',ascending=False).drop_duplicates('district').sort_values('delhi_composite_score',ascending=True)
haz_cols   = score_cols + ['delhi_composite_score']
haz_labels = ['Drought','Flood','Heatwave','Air Quality','Water Scarcity','COMPOSITE']

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(latest[haz_cols].values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(haz_labels)))
ax.set_xticklabels(haz_labels, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(latest)))
ax.set_yticklabels(latest['district'].values, fontsize=9)
for i in range(len(latest)):
    for j, col in enumerate(haz_cols):
        val = latest.iloc[i][col]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                color='white' if val > 55 else 'black', fontsize=8, fontweight='bold')
plt.colorbar(im, ax=ax, label='Risk Score (0–100)')
ax.set_title('Delhi Districts — Multi-Hazard Climate Risk (Latest Year)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_composite_heatmap.png', dpi=150)
plt.show()
print('Saved delhi_composite_heatmap.png')

## Interactive Folium Map

In [ ]:
CAT_COLORS = {'LOW':'#2ecc71','MEDIUM':'#f39c12','HIGH':'#e67e22','VERY HIGH':'#e74c3c'}

m = folium.Map(location=[28.63, 77.22], zoom_start=11, tiles='CartoDB positron')
for _, row in latest.iterrows():
    cat   = str(row['delhi_composite_category'])
    color = CAT_COLORS.get(cat, '#888')
    popup_html = f"""
    <div style='font-family:Arial; min-width:220px;'>
        <h4 style='margin:0 0 6px 0;'>{row['district']}</h4>
        <table style='font-size:12px; border-collapse:collapse; width:100%;'>
            <tr><td><b>Composite</b></td><td><b style='color:{color};'>{row['delhi_composite_score']:.1f} — {cat}</b></td></tr>
            <tr><td>Drought</td><td>{row['drought_risk_score']:.1f}</td></tr>
            <tr><td>Flood</td><td>{row['flood_risk_score']:.1f}</td></tr>
            <tr><td>Heatwave</td><td>{row['heatwave_risk_score']:.1f}</td></tr>
            <tr><td>Air Quality</td><td>{row['airquality_risk_score']:.1f}</td></tr>
            <tr><td>Water Scarcity</td><td>{row['waterscarcity_risk_score']:.1f}</td></tr>
        </table>
    </div>"""
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=min(max(row['delhi_composite_score']/6, 8), 22),
        color=color, fill=True, fill_color=color, fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row['district']}: {row['delhi_composite_score']:.1f} ({cat})"
    ).add_to(m)

map_path = f'{OUTPUT_DIR}/delhi_district_map.html'
m.save(map_path)
print(f'Saved delhi_district_map.html')
m  # Display in notebook

## Bank Recommendation Function

In [ ]:
def delhi_recommendation(row):
    cat = str(row['delhi_composite_category'])
    scores = {
        'Flood':          float(row.get('flood_risk_score', 0)),
        'Air Quality':    float(row.get('airquality_risk_score', 0)),
        'Heatwave':       float(row.get('heatwave_risk_score', 0)),
        'Water Scarcity': float(row.get('waterscarcity_risk_score', 0)),
        'Drought':        float(row.get('drought_risk_score', 0)),
    }
    top2 = sorted(scores, key=scores.get, reverse=True)[:2]
    base = {
        'LOW':       'Standard processing. Annual climate review recommended.',
        'MEDIUM':    'Climate addendum required. Review flood plain and AQI zone.',
        'HIGH':      'Climate risk insurance mandatory. Flag for environmental due diligence.',
        'VERY HIGH': 'Do not approve without independent climate assessment and insurance.',
    }[cat]
    return base + f' Primary risks: {top2[0]} and {top2[1]}.'

# Apply to 2023 data
latest_2023 = comp[comp['year']==2023].copy()
if len(latest_2023) > 0:
    latest_2023['recommendation'] = latest_2023.apply(delhi_recommendation, axis=1)
    for _, row in latest_2023.sort_values('delhi_composite_score', ascending=False).iterrows():
        print(f"\n{row['district']} ({row['delhi_composite_score']:.1f} — {row['delhi_composite_category']})")
        print(f"  → {row['recommendation']}")
else:
    print('No 2023 data. Run composite after all modules are built.')